In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/unconv.csv")

In [3]:
df

,Well,Por,Perm,AI,Brittle,TOC,VR,Prod
0,1,12.08,2.92,2.80,81.40,1.16,2.31,4165.196191
1,2,12.38,3.53,3.22,46.17,0.89,1.88,3561.146205
2,3,14.02,2.59,4.01,72.80,0.89,2.72,4284.348574
3,4,17.67,6.75,2.63,39.81,1.08,1.88,5098.680869
4,5,17.52,4.57,3.18,10.94,1.51,1.90,3406.132832
...,...,...,...,...,...,...,...,...
195,196,11.95,3.13,2.97,67.18,0.80,2.06,3847.571003
196,197,17.99,9.87,3.38,44.32,0.98,2.08,5601.227131
197,198,12.12,2.27,3.52,57.07,-0.04,1.73,3409.575363
198,199,15.55,4.48,2.48,58.25,1.89,2.35,5087.592149


In [4]:
# Списки email-адресов для вебинаров
programming_emails = {
    "bennet@xyz.com", "darcy@abc.com", "margaret@xyz.com", "pa@hhh.com",
    "marimari@xyz.com", "mallika@yahoo.com", "abc@xyz.com", "0071235@gmail.ru"
}

ml_emails = {
    "marimari@xyz.com", "darcy@abc.com", "0071235@gmail.ru",
    "darcy@abc.com", "petr44@xyz.com", "katrin@ya.com"
}

# Участники обоих вебинаров
both_webinars = programming_emails & ml_emails

# Все уникальные слушатели
all_listeners = programming_emails | ml_emails

# Только один вебинар: симметрическая разность
only_one_webinar = programming_emails ^ ml_emails

# Вывод результатов
print("Количество слушателей вебинара по программированию:", len(programming_emails))
print("Количество слушателей вебинара по машинному обучению:", len(ml_emails))
print("Слушатели, записанные на оба вебинара:", len(both_webinars))
print("Слушатели, записанные хотя бы на один вебинар:", len(all_listeners))
print("Слушатели, записанные только на один вебинар:", len(only_one_webinar))


Количество слушателей вебинара по программированию: 8
Количество слушателей вебинара по машинному обучению: 5
Слушатели, записанные на оба вебинара: 3
Слушатели, записанные хотя бы на один вебинар: 10
Слушатели, записанные только на один вебинар: 7


In [5]:
import seaborn as sns
df = sns.load_dataset('diamonds')

In [6]:
df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)

In [7]:
df = pd.get_dummies(df, drop_first=True)

In [10]:
import numpy as np
df['carat'] = np.log(1+df['carat'])
df['price'] = np.log(1+df['price'])

In [11]:
X = df.drop(columns="price")
y = df["price"]

In [13]:
import pandas as pd
import numpy as np
import seaborn as sns # Keep import, as it's part of the original request
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

try:
    # Attempt to load the dataset directly from URL
    df = pd.read_csv('https://raw.githubusercontent.com/mwaskom/seaborn-data/master/diamonds.csv')
    print("Dataset loaded successfully from URL.")
except Exception as e:
    print(f"Failed to load dataset from URL: {e}")
    print("Please ensure your environment has network access or provide the 'diamonds.csv' file directly.")
    # Exit or handle the failure appropriately, e.g., by raising an error
    raise Exception("Could not load 'diamonds' dataset. Cannot proceed.")

# 1. (Data loaded)

# Display the first few rows and information about the DataFrame
print("\nDataFrame head:")
print(df.head())
print("\nDataFrame info:")
df.info()

# 2. Preprocess Data
# Drop specified columns
df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)
print("\nDataFrame after dropping columns:")
print(df.head())

# One-hot encode categorical features
df = pd.get_dummies(df, drop_first=True)
print("\nDataFrame after one-hot encoding:")
print(df.head())

# Log transform features
df['carat'] = np.log(1 + df['carat'])
df['price'] = np.log(1 + df['price'])
print("\nDataFrame after log transformation:")
print(df.head())

# 3. Define X and y
X = df.drop(columns="price")
y = df["price"]

# 4. Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

# 5. Hyperparameter Tuning with GridSearchCV
# Define the parameter grid
param_grid = {
    "loss": ["squared_error", "epsilon_insensitive"],
    "penalty": ["elasticnet"],
    "alpha": np.logspace(-3, 3, 10), # 10 values from 10^-3 to 10^3
    "l1_ratio": np.linspace(0, 1, 10), # 10 values from 0 to 1
    "learning_rate": ["constant"],
    "eta0": np.logspace(-4, -1, 4) # 4 values from 10^-4 to 10^-1
}

# Instantiate SGDRegressor with random_state for reproducibility
sgd = SGDRegressor(random_state=42, max_iter=10000) # Increased max_iter for potential convergence

# Instantiate GridSearchCV
# scoring='neg_mean_squared_error' to find parameters that minimize MSE
# cv=3 for 3-fold cross-validation
# n_jobs=-1 to use all available CPU cores
# verbose=1 to see progress during fitting
grid_search = GridSearchCV(estimator=sgd, param_grid=param_grid, scoring='neg_mean_squared_error', cv=3, n_jobs=-1, verbose=1)

print("\nStarting GridSearchCV...")
grid_search.fit(X_train, y_train)

# Get the best estimator
best_sgd_model = grid_search.best_estimator_

# 6. Evaluate Best Model
y_pred = best_sgd_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)

# Round MSE to three decimal places
rounded_mse = round(mse, 3)

print(f"\nBest parameters found by GridSearchCV: {grid_search.best_params_}")
print(f"Mean Squared Error on the test set (rounded to 3 decimal places): {rounded_mse}")

Dataset loaded successfully from URL.

DataFrame head:
   carat      cut color clarity  depth  table  price     x     y     z
0   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98  2.43
1   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84  2.31
2   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07  2.31
3   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23  2.63
4   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35  2.75

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   carat    53940 non-null  float64
 1   cut      53940 non-null  object 
 2   color    53940 non-null  object 
 3   clarity  53940 non-null  object 
 4   depth    53940 non-null  float64
 5   table    53940 non-null  float64
 6   price    53940 non-null  int64  
 7   x        53940 non-null  float64
 8   y       

In [14]:
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import seaborn as sns
import pandas as pd
import numpy as np

df = sns.load_dataset('diamonds')

df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)
df = pd.get_dummies(df, drop_first=True)

df['carat'] = np.log(1+df['carat'])
df['price'] = np.log(1+df['price'])

X_cols = [col for col in df.columns if col!='price']
X = df[X_cols]
y = df['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

parameters = {
    "loss": ["squared_error", "epsilon_insensitive"],
    "penalty": ["elasticnet"],
    "alpha": np.logspace(-3, 3, 10),
    "l1_ratio": np.linspace(0, 1, 10),
    "learning_rate": ["constant"],
    "eta0": np.logspace(-4, -1, 4)
}

sgd = SGDRegressor(random_state=42)
sgd_cv = GridSearchCV(estimator=sgd, param_grid=parameters, n_jobs=-1)
sgd_cv.fit(X_train, y_train)

print(sgd_cv.best_params_)

sgd = SGDRegressor(**sgd_cv.best_params_, random_state = 42)

sgd.fit(X_train, y_train)
sgd.score(X_train, y_train) # r2
ls = sgd.predict(X_test)

round(mean_squared_error(y_test, ls), 3)

{'alpha': 0.001, 'eta0': 0.001, 'l1_ratio': 0.0, 'learning_rate': 'constant', 'loss': 'epsilon_insensitive', 'penalty': 'elasticnet'}


0.044

In [15]:
def func1(x):
    return 6*x**5-5*x**4-4*x**3+3*x**2
 
def func2(x):
    return 30*x**4-20*x**3-12*x**2+6*x
init_value = 0.7
iter_count = 0
x_curr = init_value
epsilon = 0.000001
f = func1(x_curr)
 
while (abs(f) > epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(x_curr)
print(iter_count)

0.6296335078534031
0.6286680781673306
0.6286669787778999
0.6286669787764609
4


In [16]:
import numpy as np

# Уравнение: x^2 + 4 = 0
# x^2 = -4
x_squared = -4

# Вычисляем корни
# NumPy может возвращать комплексные числа для отрицательных квадратных корней
x_roots = np.sqrt(x_squared)

print(f"Корни уравнения: {x_roots}, {-x_roots}")
# Вывод будет примерно таким: Корни уравнения: 2j, -2j

Корни уравнения: nan, nan


C:\Users\mi\AppData\Local\Temp\ipykernel_1232\3114738619.py:9: RuntimeWarning: invalid value encountered in sqrt
  x_roots = np.sqrt(x_squared)


In [17]:
def func(x):
    return 8*x**3-2*x**2-450
def func1(x):
    return 24*x**2 - 4*x 
def func2(x):
    return 48*x -4
 
init_value = 42
iter_count = 0
x_curr = init_value
epsilon = 0.0001
f = func1(x_curr)
 
while (abs(f) > epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(x_curr)
 
print(round(x_curr, 3))
print(round(func(x_curr),3))

21.041749502982107
10.562707090133793
5.323351550447383
2.7040050774153417
1.3949941413301903
0.7418109325525483
0.41784523900811205
0.26096925221473555
0.19169814030401197
0.16955770984744145
0.1667151339969682
0.1666666807529666
0.16666666666666785
0.167
-450.019


In [18]:
import numpy as np
from scipy.optimize import minimize

def func(x):
    return x[0] ** 2 - x[0] * x[1] + x[1] ** 2 + 9 * x[0] - 6 * x[1] + 20

def grad_func(x):
    return np.array([2 * x[0] - x[1] + 9, -x[0] + 2 * x[1] - 6])

x_0 = [-400, -400]
result = minimize(func, x_0, method='BFGS', jac=grad_func)
solution = result['x']
print(solution)

[-4.  1.]


In [19]:
from scipy.optimize import minimize

def func(x):
    return x[0]**2.0 - 3*x[0] + 45

def grad_func(x):
    return 2*x[0]-3

x_0 = 10
result = minimize(func, x_0, method='BFGS', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации Optimization terminated successfully.
Количество оценок: 5
Решение: f([1.5]) = 42.75000


In [20]:
x_0 = 10
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Количество оценок: 3
Решение: f([1.5]) = 42.75000


In [21]:
def func(x):
    return x[0]**4.0 + 6*x[1]**2.0 + 10
 
def grad_func(x):
    return np.array([4* x[0] ** 3, 12* x[1]])

x_0 = [100.0, 100.0]
result = minimize(func, x_0, method='BFGS', jac=grad_func)
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s) = %.5f' % (solution, evaluation))

Статус оптимизации Optimization terminated successfully.
Количество оценок: 37
Решение: f([1.31617159e-02 6.65344582e-14]) = 10.00000
